<a href="https://colab.research.google.com/github/Andorryu/KU_EECS_836_Final_Project/blob/main/Advanced/BEHRT_Application_to_EHR_Health_Data_CURR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INITIALIZATION: Defining Dataset, Model, Helper Functions

## Installing Dependencies, Creating Dataset using MIMIC-III

In [ ]:
# we use pyhealth for data related tasks and optuna for our hyperparameter fine-tuning
!pip install pyhealth

In [ ]:
!pip install optuna

For this section, can be changed in any way to load the MIMIC-III dataset in preferred manner

In [ ]:
# mount drive containing the downloaded MIMIC-III dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pyhealth.datasets import MIMIC3Dataset

# tables we extract data from are the ones for diagnoses, procedures, prescriptions, and lab events
dataset = MIMIC3Dataset(
    root='/content/drive/MyDrive/mimic-iii-clinical-database-1.4',
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS", "LABEVENTS"],
    dev= False
)

dataset.stats()

## Helper Functions

Functions used for extracting data for our custom tasks (age, time gap, building code vocab, type vocab, etc.) and for our hyperparameter fine-tuning

In [ ]:
from pyhealth.data import Patient, Event
from datetime import datetime
from pyhealth.models import BaseModel
from pyhealth.models.embedding import EmbeddingModel
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.trainer import Trainer
from pyhealth.utils import set_seed
from tqdm import tqdm
from typing import List, Dict
import torch
import torch.nn as nn
import random
import numpy as np
import math
import optuna

# getting date-of birth
def getDOB(P: Patient):
  return P.get_events(event_type='patients')[0].attr_dict['dob']

# getting discharge time
def getDischTime(E: Event):
  return E.attr_dict['dischtime']

# converting string to datetime
def convertToDT(dtString: str):
  format = '%Y-%m-%d %H:%M:%S'
  dt = datetime.strptime(dtString, format)
  return dt

# get age of patient at admission
def getAge(dob: datetime, admissionDate: datetime):
  age = (admissionDate - dob).days // 365
  return age

# getting hospital admission ID
def getHADID(E: Event):
  return E.attr_dict['hadm_id']

# bucketing age into decades using integer division
def bucket_age(age):
  return age // 10

# bucketing time gap based on week, month, 3 months, less than 1 year, or more than 1 year
def get_time_gap_bucket(days):
    if days < 7:
        return "gap_0_7"
    elif days < 30:
        return "gap_7_30"
    elif days < 90:
        return "gap_30_90"
    elif days < 365:
        return "gap_90_365"
    else:
        return "gap_365_plus"

# building code vocabulary, each unique code from all patient data extracted from MIMIC-III is assigned an integer value
def build_code_vocab(dataset):
  code_vocab = {
      # Padding and CLS tokens are added as well so model can recognize them
      "[PAD]": 0,
      "[CLS]": 1,
  }

  # iterating through patients and getting all clinical codes from them (diagnoses, procedures, prescriptions, and lab events)
  idx = 2
  seen = set()
  for patient_id in tqdm(dataset.unique_patient_ids, desc="building code vocab"):
      patient = dataset.get_patient(patient_id)

      diagnoses = patient.get_events(event_type="diagnoses_icd")
      procedures = patient.get_events(event_type="procedures_icd")
      medications = patient.get_events(event_type="prescriptions")
      labs = patient.get_events(event_type="labevents")

      # each code is prefixed with its code type as well for better separation of codes within the shared code vocabulary (ex. DIAG_ for diagnosis)
      for diag in diagnoses:
        raw_code = diag.attr_dict["icd9_code"]
        if raw_code is None:
          continue
        code = "DIAG_" + diag.attr_dict["icd9_code"]
        if code not in code_vocab:
          code_vocab[code] = idx
          idx += 1

      for proc in procedures:
        raw_code = proc.attr_dict["icd9_code"]
        if raw_code is None:
          continue
        code = "PROC_" + proc.attr_dict["icd9_code"]
        if code not in code_vocab:
          code_vocab[code] = idx
          idx += 1

      for med in medications:
        raw_code = med.attr_dict["drug"]
        if raw_code is None:
          continue
        code = "DRUG_" + str(raw_code)
        if code not in code_vocab:
          code_vocab[code] = idx
          idx += 1

      for lab in labs:
        raw_itemid = lab.attr_dict["itemid"]
        if raw_itemid is None:
            continue
        itemid = str(raw_itemid)
        if itemid not in seen:
          seen.add(itemid)
          code = "LAB_" + itemid
          code_vocab[code] = idx
          idx += 1

  return code_vocab

# type vocabulary, each unique clinical code is given an integer value based on type
type_vocab = {
  "PAD": 0,
  "CLS": 1,
  "diagnosis": 2,
  "procedure": 3,
  "drug": 4,
  "lab": 5,
}

# optuna objective function for hyperparameter tuning, hyperparameters can be changed by optuna for each run, optimization is validated by AUPRC
def optimize_task(trial):

    # hyperparameters that are being tuned: embedding dimensions, number of multi-attention heads, number of hidden layers, dropout, learning rate, and batch size
    embedding_dim = trial.suggest_categorical("embedding_dim", [64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    num_layers = trial.suggest_categorical("num_layers", [1, 2])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    train_dataloader = get_dataloader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_dataloader = get_dataloader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    model = BEHRT(
        dataset=sample_dataset,
        embedding_dim=embedding_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        dropout=dropout,
        max_len=1096,
    ).to(device)

    trainer = Trainer(model=model, device=device)

    # for task 1, we need per-sample AUPRC, so the metric we look at changes based on task here
    label_key = model.label_key
    if label_key == "drugs":
        metric_name = "pr_auc_samples"
    else:
        metric_name = "pr_auc"

    best_pr_auc = 0

    # each epoch is run separately so that values can be verified and if needed, trial can be pruned early
    for epoch in range(5):
        trainer.train(
            train_dataloader=train_dataloader,
            val_dataloader=val_dataloader,
            epochs=1,
            monitor=metric_name,
            optimizer_params={"lr": lr},
        )

        val_result = trainer.evaluate(val_dataloader)
        pr_auc = val_result[f"{metric_name}"]

        if pr_auc is None or not np.isfinite(pr_auc):
            raise optuna.TrialPruned()

        trial.report(pr_auc, step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()

        # return highest AUPRC value from the trial
        best_pr_auc = max(best_pr_auc, pr_auc)

    return best_pr_auc

# pruning function for optuna, using a median pruner, trials are not pruned before trial 5, and trials cannot be pruned before the first epoch of each trial
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=1,
    interval_steps=1
)

### Helpful Input Files:
Some of the preprocessing steps are very time consuming. We decided to output some of the necessary input files that can be run directly without the need to remake them, saving a lot of runtime.

* code_vocab.json - The prebuilt file containing the vocabulary for all codes contained in the patient data, each unique code is mapped to an integer value, which is used by each custom task to convert the codes into a useful format for our BEHRT model
* task1_optuna.db, task2_optuna.db, task3_optuna.db - optuna database files containing the history of trials for each hyperparameter tuning (for tasks 1, 2, and 3 respectively) can be used by optuna to get the best parameters without needing to do the hyperparameter optimization again

usage for all files: place within the google drive in this path "/content/drive/MyDrive/{the file name}" or feel free to change the filepaths so that it points to each file in the relevant code blocks.



## Model Definition (BEHRT)



In [ ]:
# positional embedding taken from BEHRT, uses a sinusodial function to denote the relevancy scores of each position
class CustomPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i]     = math.sin(pos / (10000 ** ((2 * i) / d_model)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * (i + 1)) / d_model)))
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

In [ ]:
class BEHRT(BaseModel):
  def __init__(self, dataset, embedding_dim=128, num_heads=4,
                 num_layers=2, dropout=0.5, max_len=512, use_age=True, use_time_gap=True, use_code_type=True, **kwargs):
        super().__init__(dataset=dataset, **kwargs)

        # function parameters
        # label_key specifies the type of task
        # use_<embedding> variables determine whether each embedding will be added to final embedding
        self.label_key = self.label_keys[0]
        self.use_age = use_age
        self.use_time_gap = use_time_gap
        self.use_code_type = use_code_type

        # we only use BCEWithLogitsLoss in the case of readmission
        if self.label_key == "readmission":
            labels = [dataset[i][self.label_key].item() for i in range(len(dataset))]
            num_pos = sum(labels)
            num_neg = len(labels) - num_pos
            pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32)
            self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        else:
            self.loss_fn = None

        # determining vocabulary sizes for each embedding type
        code_embed_size = dataset.input_processors["codes"].vocab_size()
        age_embed_size = dataset.input_processors["ages"].vocab_size()
        gap_embed_size = dataset.input_processors["time_gaps"].vocab_size()
        type_embed_size = dataset.input_processors["code_types"].vocab_size()

        # cls token for patient-level sequence representation
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embedding_dim))
        nn.init.normal_(self.cls_token, std=0.02)

        # initializing embedding layers
        self.code_embedding = nn.Embedding(code_embed_size, embedding_dim, padding_idx=0)
        self.type_embedding = nn.Embedding(type_embed_size, embedding_dim, padding_idx=0)
        self.age_embedding = nn.Embedding(age_embed_size, embedding_dim, padding_idx=0)
        self.gap_embedding = nn.Embedding(gap_embed_size, embedding_dim, padding_idx=0)
        self.pos_embedding = CustomPositionalEmbedding(d_model=embedding_dim, max_len=max_len+1)
        self.value_projection = nn.Linear(1, embedding_dim)

        # transformer encoder
        encoder = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        # defining transformer encoder
        self.transformer = nn.TransformerEncoder(encoder_layer= encoder, num_layers=num_layers)
        self.fc = nn.Linear(embedding_dim, self.get_output_size())


  def forward(self, **kwargs):
    device = next(self.parameters()).device
    y_true = kwargs[self.label_key].to(device)

    # all relevant inputs from custom tasks are taken in the forward pass
    codes = kwargs["codes"].to(device)
    code_types = kwargs["code_types"].to(device)
    ages = kwargs["ages"].to(device)
    time_gaps = kwargs["time_gaps"].to(device)

    batch, max_visits, max_codes = codes.shape
    seq_len = max_codes * max_visits

    # match the dimensionality of the tensors
    codes_flat = codes.reshape(batch, seq_len)
    ages_flat = ages.unsqueeze(-1).expand(-1, -1, max_codes).reshape(batch, seq_len)
    gaps_flat = time_gaps.unsqueeze(-1).expand(-1, -1, max_codes).reshape(batch, seq_len)
    types_flat = code_types.reshape(batch, seq_len)

    # flatten embeddings
    codes_embedding = self.code_embedding(codes_flat)
    types_embedding = self.type_embedding(types_flat)
    ages_embedding = self.age_embedding(ages_flat)
    gaps_embedding = self.gap_embedding(gaps_flat)

    # combining embeddings into final feature representation
    x = codes_embedding

    if self.use_age:
      x = x + ages_embedding

    if self.use_time_gap:
      x = x + gaps_embedding

    if self.use_code_type:
      x = x + types_embedding

    if "values" in kwargs:
      values = kwargs["values"].to(device).float()
      values_flat = values.reshape(batch, seq_len)
      value_embedding = self.value_projection(values_flat.unsqueeze(-1))
      x = x + value_embedding

    cls_tokens = self.cls_token.expand(batch, 1, -1)       # [B, 1, D]
    x = torch.cat([cls_tokens, x], dim=1)                  # [B, seq_len + 1, D]

    # finally we apply positional embedding
    x = self.pos_embedding(x)

    # padding mask for model
    token_padding_mask = codes_flat == 0

    cls_padding_mask = torch.zeros(
        batch,
        1,
        dtype=torch.bool,
        device=device,
    )                                                      # [B, 1]

    padding_mask = torch.cat(
        [cls_padding_mask, token_padding_mask],
        dim=1,
    )

    h = self.transformer(x, src_key_padding_mask=padding_mask)
    # h: (batch, seq_len + 1, embedding_dim)

    cls_h = h[:, 0, :]                  # (batch, embedding_dim) — pull CLS
    logit = self.fc(cls_h)

    # determines which loss function to use based on task specified
    if self.label_key == "readmission":
      loss = self.loss_fn.to(device)(logit, y_true.float())
    else:
      loss = self.get_loss_function()(logit, y_true)

    # returning required output for PyHealth evaluation function
    return {
        "loss": loss,
        "y_prob": self.prepare_y_prob(logit) ,
        "y_true": y_true,
        "logit": logit,
    }



## Creating vocabulary for all EHR codes, so that they can be mapped to numerical values

In [ ]:
import os
import json

# filepath for the code vocabulary file, both for access and file creation
vocab_path = "/content/drive/MyDrive/code_vocab.json"

# checks if file has already been created, if it has then run the pre-existing file as the code_vocab
if os.path.exists(vocab_path):
    with open(vocab_path, "r") as f:
        code_vocab = json.load(f)
# if not, then build the code vocabulary from the dataset and then output as a json file for future use
else:
    code_vocab = build_code_vocab(dataset)
    with open(vocab_path, "w") as f:
        json.dump(code_vocab, f, indent=2)

# Task 1: Task Definition, Hyperparameters, Metrics, etc.

## Task 1 Custom Task

Custom task for Task 1: Drug Recommendation. Collects relevant patient data for Task 1 and specifies input and output features needed for drug recommendation.

In [ ]:
from collections import defaultdict
from pyhealth.tasks import BaseTask
from pyhealth.data import Patient
from typing import List, Dict
from pyhealth.processors import BinaryLabelProcessor, SequenceProcessor, NestedSequenceProcessor, MultiLabelProcessor
import polars as pl

class BEHRTTask1(BaseTask):
    task_name = "BEHRT collection for task 1"
    def __init__(self, code_vocab, type_vocab):

      # input and output schema for this custom task 1, we take all clinical codes (drugs, dignoses, and procedures),
      # input for their code types, as well as the age of the patient at each visit and time gaps between visits
      super().__init__()
      self.code_vocab = code_vocab
      self.type_vocab = type_vocab
      self.input_schema = {
          "codes": NestedSequenceProcessor,
          "code_types": NestedSequenceProcessor,
          "ages": SequenceProcessor,
          "time_gaps": SequenceProcessor,
      }
      self.output_schema = {
          "drugs": MultiLabelProcessor
      }

    # pre filtering patients:
    # - patient must have enough visits, at least 2 or more
    # - patient must have diagnoses, procedures, and prescriptions
    # - patient must have a date of birth so that we can calculate age
    def pre_filter(self, df: pl.LazyFrame) -> pl.LazyFrame:

      has_enough_visits = (
            df.filter(pl.col("event_type") == "admissions")
            .group_by("patient_id")
            .agg(pl.count().alias("visit_count"))
            .filter(pl.col("visit_count") >= 2)
            .select("patient_id")
            .collect()
            .to_series()
      )

      has_diagnoses = (
          df.filter(pl.col("event_type") == "diagnoses_icd")
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      has_dob = (
          df.filter(
              (pl.col("event_type") == "patients")
              & (pl.col("patients/dob").is_not_null())
          )
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      has_prescriptions = (
          df.filter(pl.col("event_type") == "prescriptions")
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      has_procedures = (
          df.filter(pl.col("event_type") == "procedures_icd")
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      condition = (
          pl.col("patient_id").is_in(has_enough_visits)
          & pl.col("patient_id").is_in(has_diagnoses)
          & pl.col("patient_id").is_in(has_dob)
          & pl.col("patient_id").is_in(has_procedures)
          & pl.col("patient_id").is_in(has_prescriptions)
      )

      filtered_df = df.filter(condition)
      return filtered_df

    def __call__(self, patient: Patient) -> List[Dict]:
        samples = []

        dob_dt = convertToDT(getDOB(patient))

        medications = patient.get_events(event_type="prescriptions")
        diagnoses = patient.get_events(event_type='diagnoses_icd')
        procedures = patient.get_events(event_type='procedures_icd')
        admissions = patient.get_events(event_type='admissions')

        # grouping clinical codes from a patient by visit (hospital admission ID)
        visit_codes = defaultdict(list)
        for diag in diagnoses:
            raw_code = diag.attr_dict["icd9_code"]
            if raw_code is None:
              continue
            visit_codes[getHADID(diag)].append(("DIAG_" + diag.attr_dict['icd9_code'], "diagnosis"))
        for proc in procedures:
            raw_code = proc.attr_dict["icd9_code"]
            if raw_code is None:
              continue
            visit_codes[getHADID(proc)].append(("PROC_" + proc.attr_dict['icd9_code'], "procedure"))

        # grouping prescriptions from a patient by visit
        visit_medications = defaultdict(list)
        for med in medications:
            raw_code = med.attr_dict["drug"]
            if raw_code is None:
              continue
            visit_medications[getHADID(med)].append(("DRUG_" + str(raw_code), "drug"))

        # sorting visits/admissions by timestamp, earliest to latest
        sorted_admissions = sorted(admissions, key=lambda e: e.timestamp)

        # iterating through all admissions, creating a nested sequence of clinical codes grouped by visit
        for i, admission in enumerate(sorted_admissions):

            # codes and code types follow a nested structure, ages and time gaps are single value per visit
            codes_so_far = []
            types_so_far = []
            ages_so_far = []
            time_gaps_so_far = []

            # parameters for making sure the input is not too large for our model
            max_past_visits = 6 # max number of visits that will be kept in patient drug history
            max_codes_per_sample = 128 # max number of tokens/codes per each sample of a patient

            # for each visit/admission, append:
            # - all clinical codes from that visit
            # - types of each clinical code
            # - age at each visit
            # - time gap between each visit
            start_j = max(0, i - max_past_visits)
            for j in range(start_j, i + 1):
              vid = getHADID(sorted_admissions[j])
              visit_events = list(visit_codes.get(vid, []))

              if j < i:
                  visit_events += visit_medications.get(vid, [])

              visit_code_ids = []
              visit_type_ids = []
              for code, code_type in visit_events:
                if len(visit_code_ids) >= max_codes_per_sample:
                  break
                visit_code_ids.append(self.code_vocab[code])
                visit_type_ids.append(self.type_vocab[code_type])

              if len(visit_code_ids) == 0:
                  continue

              age = getAge(dob_dt, sorted_admissions[j].timestamp)
              age_bucketed = bucket_age(age)

              if j == 0:
                time_gap_bucketed = "gap_none"
              else:
                prev_discharge = convertToDT(getDischTime(sorted_admissions[j - 1]))
                curr_admit_time = sorted_admissions[j].timestamp
                gap_days = (curr_admit_time - prev_discharge).days
                time_gap_bucketed = get_time_gap_bucket(gap_days)

              codes_so_far.append(visit_code_ids)
              types_so_far.append(visit_type_ids)
              ages_so_far.append(f"age_{age_bucketed}")
              time_gaps_so_far.append(time_gap_bucketed)

            # we remove prefix for current visit drugs so that the prediction will match the actual code
            current_vid = getHADID(admission)
            current_visit_drugs = [
                code.replace("DRUG_", "")
                for code, _ in visit_medications.get(current_vid, [])
            ]

            # create dataset of samples from patient visit sequences
            samples.append({
              "patient_id": patient.patient_id,
              "codes": codes_so_far,
              "code_types": types_so_far,
              "ages": ages_so_far,
              "time_gaps": time_gaps_so_far,
              "drugs": current_visit_drugs
            })

        return samples

## Hyperparameter Tuning and Task 1 Model Metrics

In [ ]:
task = BEHRTTask1(
    code_vocab=code_vocab,
    type_vocab=type_vocab,
)
sample_dataset = dataset.set_task(task)

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)
num_trials = 20

train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset, [0.8, 0.1, 0.1], seed=42
)

study_task1 = optuna.create_study(
    direction="maximize",
    study_name="task1_behrt",
    storage="sqlite:////content/drive/MyDrive/task1_optuna.db",
    load_if_exists=True,
    pruner=pruner,
)

if len(study_task1.trials) < num_trials:
    study_task1.optimize(optimize_task, n_trials=num_trials)
else:
    print("Study exists, using pre-existing study")
best = study_task1.best_params

with open("/content/drive/MyDrive/best_task1_params.json", "w") as f:
    json.dump(study_task1.best_params, f)

In [ ]:
# running evaluation with the best hyperparameters found from our optuna study for task 1
train_dataloader = get_dataloader(
    train_dataset,
    batch_size=best["batch_size"],
    shuffle=True,
)

val_dataloader = get_dataloader(
    val_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
)

test_dataloader = get_dataloader(
    test_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
)

model = BEHRT(
    dataset=sample_dataset,
    embedding_dim=best["embedding_dim"],
    num_heads=best["num_heads"],
    num_layers=best["num_layers"],
    dropout=best["dropout"],
    max_len=1096,
).to(device)

trainer = Trainer(model=model, device=device)

trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=10,
    monitor="pr_auc_samples",
    patience=3, # we have patience here set to 3 so that a failing epoch will be cut short if needed
    optimizer_params={"lr": best["lr"]},
)

result = trainer.evaluate(test_dataloader)
print(result)

# precision@k and recall@k metrics
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_dataloader:
        output = model(**batch)

        y_true.append(output["y_true"].cpu())
        y_pred.append(output["y_prob"].cpu())

y_true = torch.cat(y_true)
y_pred = torch.cat(y_pred)

def precision_at_k(y_true, y_pred, k):
  topk = torch.topk(y_pred, k, dim=1).indices

  precisions = []

  for i in range(y_true.shape[0]):
      true_set = set(torch.where(y_true[i] == 1)[0].tolist())
      pred_set = set(topk[i].tolist())

      precisions.append(len(true_set & pred_set) / k)

  return sum(precisions) / len(precisions)

def recall_at_k(y_true, y_pred, k):
  topk = torch.topk(y_pred, k, dim=1).indices

  recalls = []

  for i in range(y_true.shape[0]):
      true_set = set(torch.where(y_true[i] == 1)[0].tolist())
      pred_set = set(topk[i].tolist())

      if len(true_set) == 0:
          recalls.append(0)
      else:
          recalls.append(len(true_set & pred_set) / len(true_set))

  return sum(recalls) / len(recalls)

print("Precision@10:", precision_at_k(y_true, y_pred, 10))
print("Recall@10:", recall_at_k(y_true, y_pred, 10))

print("Precision@20:", precision_at_k(y_true, y_pred, 20))
print("Recall@20:", recall_at_k(y_true, y_pred, 20))

### Task 1 Error Case Study: Metrics Grouped by Number of Visits

Getting Precision@k and Recall@k metrics from Task 1 results, grouping each patient by the number of visits in their clinical history.

In [ ]:
import pandas as pd
import torch

def visit_group(n):
    if n <= 2:
        return "1-2 visits"
    elif n <= 5:
        return "3-5 visits"
    else:
        return "6+ visits"

groups = [visit_group(len(sample["codes"])) for sample in test_dataset]

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_dataloader:
        output = model(**batch)
        y_true.append(output["y_true"].cpu())
        y_pred.append(output["y_prob"].cpu())

y_true = torch.cat(y_true)
y_pred = torch.cat(y_pred)

rows = []

for group in sorted(set(groups)):
    idx = [i for i, g in enumerate(groups) if g == group]

    group_y_true = y_true[idx]
    group_y_pred = y_pred[idx]

    rows.append({
        "Group": group,
        "N": len(idx),
        "Precision@10": precision_at_k(group_y_true, group_y_pred, 10),
        "Recall@10": recall_at_k(group_y_true, group_y_pred, 10),
        "Precision@20": precision_at_k(group_y_true, group_y_pred, 20),
        "Recall@20": recall_at_k(group_y_true, group_y_pred, 20),
    })

task1_error_df = pd.DataFrame(rows)
display(task1_error_df)

task1_error_df.to_csv("/content/drive/MyDrive/task1_error_analysis_by_visit_count.tsv", sep="\t", index=False)

## Task 1: Ablation Study

In [ ]:
# function for running ablations for task 1:
# - includes precision@k and recall@k metrics which only task 1 uses, as well as the per-sample AUPRC
# - input parameters specify the ability to change whether model uses age or time gap embeddings
def run_final_model(name, use_age=True, use_time_gap=True):
    set_seed(42)

    train_dataloader = get_dataloader(
    train_dataset,
    batch_size=best["batch_size"],
    shuffle=True,
    )

    val_dataloader = get_dataloader(
        val_dataset,
        batch_size=best["batch_size"],
        shuffle=False,
    )

    test_dataloader = get_dataloader(
        test_dataset,
        batch_size=best["batch_size"],
        shuffle=False,
    )

    set_seed(42)
    model = BEHRT(
        dataset=sample_dataset,
        embedding_dim=best["embedding_dim"],
        num_heads=best["num_heads"],
        num_layers=best["num_layers"],
        dropout=best["dropout"],
        max_len=1096,
        use_age=use_age,
        use_time_gap=use_time_gap,
    ).to(device)

    trainer = Trainer(model=model, device=device)

    trainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=10,
        monitor="pr_auc_samples",
        patience=3,
        optimizer_params={"lr": best["lr"]},
    )

    result = trainer.evaluate(test_dataloader)

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_dataloader:
            output = model(**batch)
            y_true.append(output["y_true"].cpu())
            y_pred.append(output["y_prob"].cpu())

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)

    result["precision@10"] = precision_at_k(y_true, y_pred, 10)
    result["recall@10"] = recall_at_k(y_true, y_pred, 10)
    result["precision@20"] = precision_at_k(y_true, y_pred, 20)
    result["recall@20"] = recall_at_k(y_true, y_pred, 20)

    print(name, result)
    return result

In [ ]:
# comparing results for ablations of age, time gap, and both age + time gap against full model
results = {}

results["full"] = run_final_model(
    "Full model",
    use_age=True,
    use_time_gap=True,
)

results["no_age"] = run_final_model(
    "No age embedding",
    use_age=False,
    use_time_gap=True,
)

results["no_time_gap"] = run_final_model(
    "No time-gap embedding",
    use_age=True,
    use_time_gap=False,
)

results["no_age_no_time_gap"] = run_final_model(
    "No age + no time-gap",
    use_age=False,
    use_time_gap=False,
)

for name, result in results.items():
    print(
        name,
        "AUPRC:", result["pr_auc_samples"],
        "Loss:", result["loss"],
    )

# Task 2: Task Definition, Trainer, Hyperparameters, Metrics

## Task 2 Custom Task

In [ ]:
from pyhealth.data import Patient, Event
from datetime import datetime

def getDOB(P: Patient):
  return P.get_events(event_type='patients')[0].attr_dict['dob']

def convertToDT(dt):
  if isinstance(dt, datetime):
        return dt
  if dt is None:
      return None
  dt = str(dt)
  for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
      try:
          return datetime.strptime(dt, fmt)
      except ValueError:
          pass

  raise ValueError(f"Could not parse datetime: {dt}")

def getAge(dob: datetime, admissionDate: datetime):
  age = (admissionDate - dob).days // 365
  return age

In [ ]:
from pyhealth.data import Patient
from datetime import datetime, timedelta
from pyhealth.tasks import BaseTask
import polars as pl
from typing import Any
from pyhealth.processors import NestedSequenceProcessor, SequenceProcessor, BinaryLabelProcessor

class BEHRTMortalityPrediction(BaseTask):
  task_name: str = "BEHRTMortalityPrediction"

  """
    codes -> [[labitemID1], [labitemID2], ...]] # each labitem is associated with a labevent
    values -> [[value1], [value2], ...]] # each value is associated with its corresponding labitemID
    code_types -> [['lab'], ['lab'], ...]]
    ages -> [age, age, age, ...] # one element per labevent, same age throughout (binned per decade)
    time_gaps -> [time_gap1, time_gap2, ...] # time_gap(N) represents time gap between labitemID(N-1) and labitemID(N), (binned per hour)
  """

  input_schema: dict[str, str] = {
      "codes": NestedSequenceProcessor,
      "code_types": NestedSequenceProcessor,
      "values": NestedSequenceProcessor,
      "ages": SequenceProcessor,
      "time_gaps": SequenceProcessor
  }
  output_schema: dict[str, str] = {
      'mortality': BinaryLabelProcessor
  }

  def __init__(self, code_vocab, type_vocab):
    super().__init__()
    self.code_vocab = code_vocab
    self.type_vocab = type_vocab

  def __call__(self, patient: Patient) -> list[dict[str, Any]]:
    input_window_hours = 24 # number of hours after admission time (W)
    max_events = 256 # maximum number of events included per admission/sample,
    # 256 represents the 99th percentile of events per visit from our statistics
    samples = []
    dob = convertToDT(getDOB(patient))

    for admission in patient.get_events(event_type="admissions"):

      # skip if patient was discharged during input window
      admission_dischtime = convertToDT(admission.attr_dict["dischtime"])
      duration_hour = (
        admission_dischtime - admission.timestamp
      ).total_seconds() / 3600
      if duration_hour <= input_window_hours:
        continue

      # get labevents during input window
      predict_time = admission.timestamp + timedelta(hours=input_window_hours)
      labevents_df = patient.get_events(
        event_type="labevents",
        start=admission.timestamp,
        end=predict_time,
        return_df=True,
      )

      # misc. filtering
      labevents_df = labevents_df.filter(
        pl.col("timestamp").is_not_null()
      )
      labevents_df = labevents_df.with_columns(
        pl.col("labevents/valuenum").cast(pl.Float64, strict=False)
      )
      labevents_df = labevents_df.filter(
        pl.col("labevents/valuenum").is_not_null()
      )

      # skip if admission has invalid data
      if labevents_df.height == 0:
        continue

      # select only timestamp, itemid, and valuenum
      labevents_df = labevents_df.select(
        pl.col("timestamp"),
        pl.col("labevents/itemid"),
        pl.col("labevents/valuenum").cast(pl.Float64)
      )

      # sort
      labevents_df = labevents_df.sort("timestamp")

      age = getAge(dob, admission.timestamp)
      age_bucket = f"age_{age//10}"

      # codes, code types, values, ages
      codes = []
      code_types = []
      values = []
      ages = []
      time_gaps = []

      prev_timestamp = None
      for row in labevents_df.iter_rows(named=True):
        itemid = str(row["labevents/itemid"])
        value = float(row["labevents/valuenum"])
        timestamp = row["timestamp"]

        code = "LAB_" + itemid
        if code not in self.code_vocab:
          continue

        codes.append([self.code_vocab[code]])
        code_types.append([self.type_vocab["lab"]])
        values.append([value])
        ages.append(age_bucket)

        # time gaps
        if prev_timestamp is None:
          time_gaps.append("gap_none")
        else:
          hrs = int((timestamp - prev_timestamp).total_seconds()//3600)
          time_gaps.append(f"gap_{hrs}_{hrs+1}")
        prev_timestamp = timestamp

      # mortality
      if len(codes) == 0:
        continue
      mortality = int(admission.hospital_expire_flag)

      if len(codes) > max_events:
          codes = codes[-max_events:]
          code_types = code_types[-max_events:]
          values = values[-max_events:]
          ages = ages[-max_events:]
          time_gaps = time_gaps[-max_events:]
          time_gaps[0] = "gap_none"

      samples.append(
        {
          "patient_id": patient.patient_id,
          "admission_id": admission.hadm_id,
          "codes": codes,
          "code_types": code_types,
          "values": values,
          "ages": ages,
          "time_gaps": time_gaps,
          "mortality": mortality
        }
      )

    return samples


## Hyperparameter Tuning and Task 2 Model Performance Metrics

Hyperparameter tuning and outputting results from both model and ablation study follow the same process as in Task 1. Except for this task we include AUROC metric and exclude the Precision@k and Recall@k metrics

In [ ]:
task = BEHRTMortalityPrediction(
    code_vocab=code_vocab,
    type_vocab=type_vocab,
)
sample_dataset = dataset.set_task(task)

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)
num_trials = 20

train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset, [0.8, 0.1, 0.1], seed=42
)

study_task2 = optuna.create_study(
    direction="maximize",
    study_name="task2_behrt",
    storage="sqlite:////content/drive/MyDrive/task2_optuna.db",
    load_if_exists=True,
    pruner=pruner,
)

if len(study_task2.trials) < num_trials:
    study_task2.optimize(optimize_task, n_trials=num_trials)
else:
    print("Study exists, using pre-existing study")
best = study_task2.best_params

with open("/content/drive/MyDrive/best_task2_params.json", "w") as f:
    json.dump(study_task2.best_params, f)

In [ ]:
train_dataloader = get_dataloader(
    train_dataset,
    batch_size=best["batch_size"],
    shuffle=True,
)

val_dataloader = get_dataloader(
    val_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
)

test_dataloader = get_dataloader(
    test_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
)

model = BEHRT(
    dataset=sample_dataset,
    embedding_dim=best["embedding_dim"],
    num_heads=best["num_heads"],
    num_layers=best["num_layers"],
    dropout=best["dropout"],
    max_len=1096,
).to(device)

trainer = Trainer(model=model, device=device)

trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=10,
    monitor="pr_auc",
    patience=3,
    optimizer_params={"lr": best["lr"]},
)

result = trainer.evaluate(test_dataloader)
print(result)

### Task 2 Error Case Study: Metrics Grouped by Number of labevents

Observing AUPRC and AUROC metrics for each patient/admission grouped by the number of lab events per patient/admission. We also observe percentage distribution of the positive binary class for each subgroup

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

def lab_group(n):
    if n <= 25:
        return "low labs"
    elif n <= 100:
        return "medium labs"
    else:
        return "high labs"

groups = [lab_group(len(sample["codes"])) for sample in test_dataset]

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_dataloader:
        output = model(**batch)
        y_true.append(output["y_true"].detach().cpu())
        y_pred.append(output["y_prob"].detach().cpu())

y_true = torch.cat(y_true).numpy().ravel()
y_pred = torch.cat(y_pred).numpy().ravel()

rows = []

for group in sorted(set(groups)):
    idx = [i for i, g in enumerate(groups) if g == group]

    group_y_true = y_true[idx]
    group_y_pred = y_pred[idx]
    group_y_binary = (group_y_pred >= 0.5).astype(int)

    row = {
        "Group": group,
        "N": len(idx),
        "Positive Rate": group_y_true.mean()
    }

    if len(np.unique(group_y_true)) > 1:
        row["AUROC"] = roc_auc_score(group_y_true, group_y_pred)
        row["AUPRC"] = average_precision_score(group_y_true, group_y_pred)
    else:
        row["AUROC"] = None
        row["AUPRC"] = None

    rows.append(row)

task2_error_df = pd.DataFrame(rows)
display(task2_error_df)

task2_error_df.to_csv("/content/drive/MyDrive/task2_error_analysis_by_lab_count.tsv", sep="\t", index=False)

## Task 2: Ablation Study

In [ ]:
# ablation function for tasks 2 and 3 that use AUPRC instead of per-sample AUPRC
def run_final_model_task_2_3(name, use_age=True, use_time_gap=True):
    set_seed(42)

    train_dataloader = get_dataloader(
    train_dataset,
    batch_size=best["batch_size"],
    shuffle=True,
    )

    val_dataloader = get_dataloader(
        val_dataset,
        batch_size=best["batch_size"],
        shuffle=False,
    )

    test_dataloader = get_dataloader(
        test_dataset,
        batch_size=best["batch_size"],
        shuffle=False,
    )

    set_seed(42)
    model = BEHRT(
        dataset=sample_dataset,
        embedding_dim=best["embedding_dim"],
        num_heads=best["num_heads"],
        num_layers=best["num_layers"],
        dropout=best["dropout"],
        max_len=1096,
        use_age=use_age,
        use_time_gap=use_time_gap,
    ).to(device)

    trainer = Trainer(model=model, device=device)

    trainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=10,
        monitor="pr_auc",
        patience=3,
        optimizer_params={"lr": best["lr"]},
    )

    result = trainer.evaluate(test_dataloader)
    print(name, result)
    return result

In [ ]:
results = {}

results["full"] = run_final_model_task_2_3(
    "Full model",
    use_age=True,
    use_time_gap=True,
)

results["no_age"] = run_final_model_task_2_3(
    "No age embedding",
    use_age=False,
    use_time_gap=True,
)

results["no_time_gap"] = run_final_model_task_2_3(
    "No time-gap embedding",
    use_age=True,
    use_time_gap=False,
)

results["no_age_no_time_gap"] = run_final_model_task_2_3(
    "No age + no time-gap",
    use_age=False,
    use_time_gap=False,
)

for name, result in results.items():
    print(
        name,
        "AUPRC:", result["pr_auc"],
        "AUROC:", result["roc_auc"],
        "Loss:", result["loss"],
    )

# Task 3: Custom Task, Hyperparameters, Metrics, Debugging

## Task 3 Custom Task

In [ ]:
from collections import defaultdict
from pyhealth.tasks import BaseTask
from pyhealth.data import Patient
from typing import List, Dict
from pyhealth.processors import BinaryLabelProcessor, SequenceProcessor, NestedSequenceProcessor, MultiLabelProcessor
import polars as pl


class BEHRTTask3(BaseTask):
    task_name = "BEHRT collection task for task 3"
    MAX_VISITS = 30 # max number of visits
    MAX_CODES_PER_VISIT = 30 # max number of codes per visit
    def __init__(self, code_vocab, type_vocab):
      super().__init__()
      self.code_vocab = code_vocab
      self.type_vocab = type_vocab
      self.input_schema: Dict[str, str] = {
          "codes": NestedSequenceProcessor,
          "code_types": NestedSequenceProcessor,
          "time_gaps": SequenceProcessor,
          "ages": SequenceProcessor,
      }
      self.output_schema: Dict[str, str] = {
          "readmission": BinaryLabelProcessor,
      }

    # prefilter here includes patient not having died in the hospital during admission
    def pre_filter(self, df: pl.LazyFrame) -> pl.LazyFrame:

        has_enough_visits = (
            df.filter(pl.col("event_type") == "admissions")
            .group_by("patient_id")
            .agg(pl.count().alias("visit_count"))
            .filter(pl.col("visit_count") >= 2)
            .select("patient_id")
            .collect()
            .to_series()
        )

        has_diagnoses = (
            df.filter(pl.col("event_type") == "diagnoses_icd")
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        has_procedures = (
            df.filter(pl.col("event_type") == "procedures_icd")
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        has_dob = (
            df.filter(
                (pl.col("event_type") == "patients")
                & (pl.col("patients/dob").is_not_null())
            )
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        died_in_hospital = (
            df.filter(
                (pl.col("event_type") == "admissions")
                & (pl.col("admissions/hospital_expire_flag") == "1")
            )
            .select("patient_id")
            .unique()
            .collect()
            .to_series()
        )

        filtered_df = df.filter(
            pl.col("patient_id").is_in(has_enough_visits)
            & (pl.col("patient_id").is_in(has_diagnoses) | pl.col("patient_id").is_in(has_procedures))
            & pl.col("patient_id").is_in(has_dob)
            & ~pl.col("patient_id").is_in(died_in_hospital)
        )

        return filtered_df

    def __call__(self, patient: Patient) -> List[Dict]:
        samples = []

        dob_dt = convertToDT(getDOB(patient))

        medications = patient.get_events(event_type="prescriptions")
        diagnoses = patient.get_events(event_type='diagnoses_icd')
        procedures = patient.get_events(event_type='procedures_icd')
        admissions = patient.get_events(event_type='admissions')

        # task 3 requires diagnosis and procedure codes
        visit_codes = defaultdict(list)
        for diag in diagnoses:
            raw_code = diag.attr_dict["icd9_code"]
            if raw_code is None:
              continue
            visit_codes[getHADID(diag)].append(("DIAG_" + diag.attr_dict['icd9_code'], "diagnosis"))
        for proc in procedures:
            raw_code = proc.attr_dict["icd9_code"]
            if raw_code is None:
              continue
            visit_codes[getHADID(proc)].append(("PROC_" + proc.attr_dict['icd9_code'], "procedure"))
        for vid in visit_codes:
            visit_codes[vid] = visit_codes[vid][:self.MAX_CODES_PER_VISIT]


        sorted_admissions = sorted(admissions, key=lambda e: e.timestamp)

        sorted_admissions = sorted_admissions[-self.MAX_VISITS:]

        # adding all necessary information to our input schema. Including: clinical codes, type of codes, ages, and time gaps, etc.
        for i, admission in enumerate(sorted_admissions):
            codes_so_far = []
            types_so_far = []
            ages_so_far = []
            time_gaps_so_far = []

            for j in range(i + 1):
                vid = getHADID(sorted_admissions[j])
                visit_events = list(visit_codes.get(vid, []))

                visit_code_ids = []
                visit_type_ids = []
                for code, code_type in visit_events:
                    visit_code_ids.append(self.code_vocab[code])
                    visit_type_ids.append(self.type_vocab[code_type])

                if len(visit_code_ids) == 0:
                    continue

                codes_so_far.append(visit_code_ids)
                types_so_far.append(visit_type_ids)

                age = getAge(dob_dt, sorted_admissions[j].timestamp)
                ages_so_far.append(f"age_{bucket_age(age)}")

                if j == 0:
                    time_gaps_so_far.append("gap_none")
                else:
                    prev_discharge = convertToDT(getDischTime(sorted_admissions[j - 1]))
                    curr_admit = sorted_admissions[j].timestamp
                    gap_days = (curr_admit - prev_discharge).days
                    time_gaps_so_far.append(get_time_gap_bucket(gap_days))

            # readmission specific to task 3
            if i < len(sorted_admissions) - 1:
                discharge = convertToDT(getDischTime(admission))
                next_admit = sorted_admissions[i + 1].timestamp
                gap = (next_admit - discharge).days
                readmission = 1 if gap <= 30 else 0
            else:
                readmission = 0

            samples.append({
                "patient_id": patient.patient_id,
                "codes": codes_so_far,
                "code_types": types_so_far,
                "ages": ages_so_far,
                "time_gaps": time_gaps_so_far,
                "readmission": readmission,
            })

        return samples

## Hyperameter Tuning and Task 3 Model Metrics


In [ ]:
import torch
import json

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)
num_trials = 20

behrt_readmission_task = BEHRTTask3(
    code_vocab=code_vocab,
    type_vocab=type_vocab,
)

# Setting the new custom task to the dataset
sample_dataset = dataset.set_task(behrt_readmission_task)

train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset, [0.8, 0.1, 0.1], seed=42
)

study_task3 = optuna.create_study(
    direction="maximize",
    study_name="task3_behrt",
    storage="sqlite:////content/drive/MyDrive/task3_optuna.db",
    load_if_exists=True,
    pruner=pruner,
)

if len(study_task3.trials) < num_trials:
    study_task3.optimize(optimize_task, n_trials=num_trials)
else:
    print("Study exists, using pre-existing study")
best = study_task3.best_params

with open("/content/drive/MyDrive/best_task3_params.json", "w") as f:
    json.dump(study_task3.best_params, f)

In [ ]:
train_dataloader = get_dataloader(
    train_dataset,
    batch_size=best["batch_size"],
    shuffle=True,
)

val_dataloader = get_dataloader(
    val_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
)

test_dataloader = get_dataloader(
    test_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
)

model = BEHRT(
    dataset=sample_dataset,
    embedding_dim=best["embedding_dim"],
    num_heads=best["num_heads"],
    num_layers=best["num_layers"],
    dropout=best["dropout"],
    max_len=1096,
).to(device)

trainer = Trainer(model=model, device=device)

trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=10,
    monitor="pr_auc",
    patience=3,
    optimizer_params={"lr": best["lr"]},
)

result = trainer.evaluate(test_dataloader)
print(result)

### Task 3 Error Case Study: Metrics Grouped by Number of Visits

Observing AUPRC and AUROC metrics for each patient grouped by number of visits in each patient's clinical history. We also observe percentage distribution of the positive binary class for each subgroup

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

def visit_group(n):
    if n <= 2:
        return "1-2 visits"
    elif n <= 5:
        return "3-5 visits"
    else:
        return "6+ visits"

groups = [visit_group(len(sample["codes"])) for sample in test_dataset]

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_dataloader:
        output = model(**batch)
        y_true.append(output["y_true"].detach().cpu())
        y_pred.append(output["y_prob"].detach().cpu())

y_true = torch.cat(y_true).numpy().ravel()
y_pred = torch.cat(y_pred).numpy().ravel()

rows = []

for group in sorted(set(groups)):
    idx = [i for i, g in enumerate(groups) if g == group]

    group_y_true = y_true[idx]
    group_y_pred = y_pred[idx]
    group_y_binary = (group_y_pred >= 0.5).astype(int)

    row = {
        "Group": group,
        "N": len(idx),
        "Positive Rate": group_y_true.mean()
    }

    if len(np.unique(group_y_true)) > 1:
        row["AUROC"] = roc_auc_score(group_y_true, group_y_pred)
        row["AUPRC"] = average_precision_score(group_y_true, group_y_pred)
    else:
        row["AUROC"] = None
        row["AUPRC"] = None

    rows.append(row)

task3_error_df = pd.DataFrame(rows)
display(task3_error_df)

task3_error_df.to_csv("/content/drive/MyDrive/task3_error_analysis_by_visit_count.tsv", sep="\t", index=False)

## Task 3: Ablation Study

In [ ]:
# ablations for task 3 use the same ablation study function as for task 2 since they have the same metrics (AUPRC instead of Per-Sample AUPRC)
results = {}

results["full"] = run_final_model_task_2_3(
    "Full model",
    use_age=True,
    use_time_gap=True,
)

results["no_age"] = run_final_model_task_2_3(
    "No age embedding",
    use_age=False,
    use_time_gap=True,
)

results["no_time_gap"] = run_final_model_task_2_3(
    "No time-gap embedding",
    use_age=True,
    use_time_gap=False,
)

results["no_age_no_time_gap"] = run_final_model_task_2_3(
    "No age + no time-gap",
    use_age=False,
    use_time_gap=False,
)

for name, result in results.items():
    print(
        name,
        "AUPRC:", result["pr_auc"],
        "AUROC:", result["roc_auc"],
        "Loss:", result["loss"],
    )

# Exploratory Data Analysis of the MIMICIII Dataset

This section focuses on some exploratory data analysis (EDA) related to the MIMICIII dataset.


1.   Distribution of the number of events/codes that a patient can have. We chose to separate the counts by code type (proc, diag, drugs, labs).

2.   Binary class distributions for both task 2: mortality prediction and task 3: readmission prediction. This was done to determine class imbalances, which might affect the performance of each task.

3.   Distribution of the number of visits that each patient has.

## Distribution of \# of events per patient (total number of events divided into: procedures, diagnoses, prescriptions, and labs)

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

diag_counts = []
proc_counts = []
drug_counts = []
lab_counts  = []

for patient_id in tqdm(dataset.unique_patient_ids):
    patient = dataset.get_patient(patient_id)

    diag_counts.append(len(patient.get_events("diagnoses_icd")))
    proc_counts.append(len(patient.get_events("procedures_icd")))
    drug_counts.append(len(patient.get_events("prescriptions")))
    lab_counts.append(len(patient.get_events("labevents")))


df = pd.DataFrame({
    "Diagnoses": diag_counts,
    "Procedures": proc_counts,
    "Prescriptions": drug_counts,
    "Labs": lab_counts
})

print(df.describe())

means = [
    np.mean(diag_counts),
    np.mean(proc_counts),
    np.mean(drug_counts),
    np.mean(lab_counts)
]

df.to_csv("/content/drive/MyDrive/MIMICIII_event_counts.tsv", sep="\t", index=False)
df.describe().to_csv("/content/drive/MyDrive/MIMICIII_summary_statistics.tsv", sep="\t", index=False)

labels = ["Diagnoses", "Procedures", "Prescriptions", "Labs"]

plt.bar(labels, means)
plt.ylabel("Avg # events per patient")
plt.title("Average Clinical Events per Patient by Type")
plt.show()

## Binary class distributions for Task 2 and 3

In [ ]:
def binary_class_distribution(sample_dataset, label_key, task_name):
    labels = []

    for sample in sample_dataset:
        label = sample[label_key]

        if hasattr(label, "item"):
            label = label.item()

        labels.append(int(label))

    df = pd.Series(labels).value_counts().sort_index().reset_index()
    df.columns = ["Label", "Count"]
    df["Proportion"] = df["Count"] / df["Count"].sum()

    print(f"\n{task_name} class distribution")
    display(df)

    df.to_csv(f"/content/drive/MyDrive/{task_name}_class_distribution.tsv", sep="\t", index=False)

    return df


task2 = BEHRTMortalityPrediction(
    code_vocab=code_vocab,
    type_vocab=type_vocab,
)
task2_dataset = dataset.set_task(task2)

task2_dist = binary_class_distribution(
    task2_dataset,
    label_key="mortality",
    task_name="Task2_Mortality"
)

task3 = BEHRTTask3(
    code_vocab=code_vocab,
    type_vocab=type_vocab,
)
task3_dataset = dataset.set_task(task3)

task3_dist = binary_class_distribution(
    task3_dataset,
    label_key="readmission",
    task_name="Task3_Readmission"
)

## Distribution of number of visits per patient

In [ ]:
visit_counts = []

for patient_id in dataset.unique_patient_ids[:5000]:
    patient = dataset.get_patient(patient_id)
    admissions = patient.get_events(event_type="admissions")
    visit_counts.append(len(admissions))
visit_counts.sort()

for p in [50, 75, 90, 95, 99]:
    print(p, visit_counts[int(len(visit_counts) * p / 100)])